# 03 — Contingency tables

Replaces the contingency-table sections of `260419_MSEvis.ipynb` and
`260426_uvvis.ipynb`. Built around the registry: pick any number of runs
(same lead, varying predictors — or same predictors, varying lead) and
compare their joint truth/prediction histograms side-by-side.

In [ ]:
import sys, pathlib
REPO = pathlib.Path('..').resolve()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import numpy as np

from latentsky import list_runs, load_run
from latentsky.inference import predict_piece
from latentsky.plotting import build_contingency_table, plot_contingency_heatmap

list_runs()

## One run: contingency table + heatmap

In [ ]:
RUN_NAME = 'mseuv_lead3h'
PIECE = 32

run = load_run(RUN_NAME)
_, Y, Y_pred = predict_piece(run, PIECE, batch_size=4)
df, confusion, bins = build_contingency_table(Y, Y_pred, nbins=10)
df

In [ ]:
plot_contingency_heatmap(confusion, bins, title=f'{RUN_NAME} — piece {PIECE}')

## Compare runs side-by-side

Pick a list of runs (same lead, different predictor sets is the typical
comparison for this paper). Each gets one heatmap on a shared bin grid.

In [ ]:
import matplotlib.pyplot as plt

RUNS_TO_COMPARE = ['mseuv_lead3h', 'humid_lead3h', 'mse_lead3h']
PIECE = 32

# Compute predictions and the shared bin grid.
preds = {}
for n in RUNS_TO_COMPARE:
    r = load_run(n)
    _, Y, Yp = predict_piece(r, PIECE, batch_size=4)
    preds[n] = (Y, Yp)

vmax = max(max(Y.max(), Yp.max()) for Y, Yp in preds.values())
shared_bins = np.linspace(0, float(vmax), 11)

fig, axes = plt.subplots(1, len(RUNS_TO_COMPARE), figsize=(6 * len(RUNS_TO_COMPARE), 5))
for ax, name in zip(axes, RUNS_TO_COMPARE):
    Y, Yp = preds[name]
    _, conf, _ = build_contingency_table(Y, Yp, bins=shared_bins)
    centers = 0.5 * (shared_bins[:-1] + shared_bins[1:])
    ax.plot(centers, centers, 'o', mfc='none', mec='red', ms=4)
    with np.errstate(divide='ignore'):
        log_conf = np.log(np.where(conf > 0, conf, np.nan))
    im = ax.imshow(log_conf, origin='upper', aspect='auto',
                   extent=[shared_bins[0], shared_bins[-1], shared_bins[-1], shared_bins[0]])
    ax.set_title(name)
    ax.set_xlabel('predicted')
    ax.set_ylabel('truth')
    plt.colorbar(im, ax=ax, label='log(count)')
plt.tight_layout()
plt.show()